# Physics Deep Dive — hotSpring

hotSpring's most compelling domain contribution: first-principles nuclear structure
and lattice QCD on consumer GPU hardware. This notebook dives into the physics
validation arc, sovereign GPU pipeline, and the guideStone security posture.

**Data sources:** `security_convergence.json`, `benchmark_timing.json`, `test_suite_report.json`

**Reproduce:** `cargo test --lib` in `barracuda/`, individual `validate_*` binaries.

---

*For other springs:* This is hotSpring's unique domain notebook. Replace with your
most compelling discovery — the one that justifies your spring's existence in the ecosystem.

In [ ]:
import json
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

RESULTS = Path('..') / 'experiments' / 'results'

def load(name):
    with open(RESULTS / name) as f:
        return json.load(f)

security = load('security_convergence.json')
bench = load('benchmark_timing.json')
tests = load('test_suite_report.json')

print(f"guideStone Level: {security['guidestone']['level']}")
print(f"Bare checks: {security['guidestone']['bare_passed']}/{security['guidestone']['bare_checks']}")
print(f"BTSP Phase 3: {security['btsp_posture']['primals_with_phase3']}/{security['btsp_posture']['primals_total']}")
print(f"Gaps: {security['gap_summary']['resolved']} resolved / {security['gap_summary']['active']} active / {security['gap_summary']['total_gaps']} total")

## Three-Tier Validation Arc

hotSpring's validation architecture stacks three tiers:

1. **Python baselines** — Published paper reproductions (Phase A-E)
2. **Rust validation** — Same algorithms, GPU-accelerated (`cargo test --lib`)
3. **NUCLEUS IPC** — Primal composition validates IPC matches direct Rust

Each tier must agree before a science claim is trusted in production.

In [ ]:
C_PASS = '#2ecc71'
C_INFO = '#3498db'
C_GPU  = '#9b59b6'
C_PYTHON = '#f39c12'
C_RUST = '#1abc9c'
C_FAIL = '#e74c3c'

# Physics module breakdown
physics_mods = {}
for mod, data in tests['modules'].items():
    if data['tests'] >= 20:
        physics_mods[mod] = data

sorted_mods = sorted(physics_mods.items(), key=lambda x: x[1]['tests'], reverse=True)
mod_names = [m[0].replace('_', ' ').title() for m in sorted_mods]
mod_tests = [m[1]['tests'] for m in sorted_mods]
mod_times = [m[1]['time_ms'] / 1000 for m in sorted_mods]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Test distribution
axes[0].barh(mod_names, mod_tests, color=C_INFO)
axes[0].set_xlabel('Tests')
axes[0].set_title(f'Physics Modules (≥20 tests, {sum(mod_tests)} total)')
axes[0].invert_yaxis()

# Time distribution
axes[1].barh(mod_names, mod_times, color=C_GPU)
axes[1].set_xlabel('Time (seconds)')
axes[1].set_title('Module Execution Time')
axes[1].invert_yaxis()

fig.suptitle(f'hotSpring Physics: {tests["total_tests"]} Tests, {tests["total_time_s"]}s Total', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/hotspring_05_physics.png', dpi=150, bbox_inches='tight')
plt.show()

## Sovereign GPU Pipeline

hotSpring's sovereign GPU pipeline replaces nouveau with a pure Rust path:
VFIO → SovereignInit (8 stages) → native SASS/GFX compilation → dispatch.
Validated across 3 GPU generations: K80 (Kepler/SM35), Titan V (Volta/SM70),
RTX 5060 (Blackwell/SM120).

In [ ]:
gpu_generations = {
    'K80 (Kepler SM35)': {'shaders': 10, 'status': 'PROVEN', 'year': 2014},
    'Titan V (Volta SM70)': {'shaders': 10, 'status': 'PROVEN', 'year': 2017},
    'RTX 5060 (Blackwell SM120)': {'shaders': 10, 'status': 'PROVEN', 'year': 2025}
}

sovereign_stages = [
    'HBM2 Training', 'PMC Engine Gating', 'Topology Discovery',
    'PFB Memory Controller', 'Falcon Boot Chain', 'GR Engine Init',
    'PFIFO Discovery', 'GR Context Setup'
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# GPU generations
gen_names = list(gpu_generations.keys())
gen_shaders = [gpu_generations[g]['shaders'] for g in gen_names]
gen_colors = ['#f39c12', C_GPU, C_PASS]
bars = axes[0].bar(gen_names, gen_shaders, color=gen_colors)
for bar, g in zip(bars, gen_names):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                 f'{gpu_generations[g]["status"]}', ha='center', fontsize=9, fontweight='bold')
axes[0].set_ylabel('HMC Pipeline Shaders')
axes[0].set_title('Sovereign Compile Parity: 3 GPU Generations')
axes[0].tick_params(axis='x', rotation=15)

# SovereignInit pipeline stages
axes[1].barh(sovereign_stages, range(len(sovereign_stages), 0, -1),
             color=[C_RUST]*len(sovereign_stages))
axes[1].set_title('SovereignInit: 8-Stage Pure Rust Pipeline')
axes[1].set_xticks([])
axes[1].invert_yaxis()

fig.suptitle('Sovereign GPU: Zero nouveau, Zero DRM — Pure Rust + VFIO + Firmware', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/hotspring_05_sovereign.png', dpi=150, bbox_inches='tight')
plt.show()

## guideStone Security Posture

hotSpring's guideStone Level 5 certification validates 5 properties.
As a NUCLEUS consumer, BTSP Phase 3 security is inherited from the
13 primals it composes with.

In [ ]:
gs = security['guidestone']
props = gs['properties']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# guideStone properties
prop_names = list(props.keys())
prop_labels = [k.replace('_', ' ') for k in prop_names]
prop_colors = [C_PASS]*len(prop_names)
axes[0].barh(prop_labels, [1]*len(prop_names), color=prop_colors)
for i, p in enumerate(prop_names):
    detail = props[p]
    if isinstance(detail, str):
        text = detail[:60]
    else:
        text = detail.get('method', detail.get('status', 'PASS'))[:60] if isinstance(detail, dict) else str(detail)[:60]
    axes[0].text(0.05, i, text, va='center', fontsize=7, color='white', fontweight='bold')
axes[0].set_title(f'guideStone Level {gs["level"]}: All 5 Properties PASS')
axes[0].set_xticks([])
axes[0].invert_yaxis()

# Gap resolution arc
gaps = security['gap_summary']
gap_labels = ['Resolved', 'Active', 'Blocked\nUpstream']
gap_counts = [gaps['resolved'], gaps['active'], gaps['blocked_upstream']]
gap_colors = [C_PASS, '#f39c12', C_FAIL]
axes[1].bar(gap_labels, gap_counts, color=gap_colors)
for i, v in enumerate(gap_counts):
    axes[1].text(i, v + 0.5, str(v), ha='center', fontweight='bold')
axes[1].set_ylabel('Gaps')
axes[1].set_title(f'PRIMAL_GAPS: {gaps["total_gaps"]} Total ({gaps["resolved"]} Resolved)')

fig.suptitle('Security + Quality: guideStone Level 5 CERTIFIED', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/hotspring_05_security.png', dpi=150, bbox_inches='tight')
plt.show()

## Code Safety Assessment

hotSpring enforces strict code safety at the library level while allowing
controlled unsafe in feature-gated GPU binaries.

In [ ]:
safety = security['code_safety']

safety_items = [
    ('lib forbid(unsafe)', safety['lib_forbid_unsafe']),
    ('deny.toml enforced', safety['deny_toml']),
    ('dyn dispatch (prod) = 0', safety['dyn_dispatch_production'] == 0),
    ('#[allow] in prod = 0', safety['allow_attributes_production'] == 0),
    ('#[expect(lint, reason)]', True),
]

fig, ax = plt.subplots(figsize=(8, 4))
labels = [s[0] for s in safety_items]
values = [1 if s[1] else 0 for s in safety_items]
colors = [C_PASS if v else C_FAIL for v in values]

ax.barh(labels, values, color=colors)
ax.set_xlim(0, 1.5)
ax.set_xticks([])
for i, (label, val) in enumerate(safety_items):
    ax.text(1.05, i, 'YES' if val else 'NO', va='center', fontweight='bold',
            color=C_PASS if val else C_FAIL)

ax.set_title(f'Code Safety: {safety["unsafe_blocks"]} unsafe blocks (feature-gated binaries only)')
ax.invert_yaxis()

plt.tight_layout()
plt.savefig('/tmp/hotspring_05_safety.png', dpi=150, bbox_inches='tight')
plt.show()

## Validation Summary

| Domain | Highlight |
|--------|-----------|
| Nuclear EOS | **2,042 AME2020 nuclei**, L1-L3 on single RTX 4070, 1,990 novel predictions |
| Lattice QCD | **SU(3) HMC/RHMC**, gradient flow, beta-scan deconfinement at β=5.69 |
| Sovereign GPU | **3 generations** (Kepler/Volta/Blackwell), 8-stage SovereignInit, zero nouveau |
| DF64 | **3.24 TFLOPS** emulated double precision on FP32 cores |
| guideStone | **Level 5**, 30/30 bare, BLAKE3 P3, 5/5 properties |
| Code safety | **forbid(unsafe)** in lib, deny.toml, zero dyn dispatch, zero #[allow] |
| PRIMAL_GAPS | **32/43 resolved**, 8 active, 3 blocked upstream |

---

**Provenance:** All data from `experiments/results/` committed JSON artifacts.  
**Papers:** 22 published papers reproduced — see `specs/PAPER_REVIEW_QUEUE.md`.  
**Source:** [hotSpring on GitHub](https://github.com/syntheticChemistry/hotSpring) · [primals.eco](https://primals.eco/lab/springs/hotspring/)